# DEEP LEARNING FINAL PROJECT

Team Member:
- Leonardus Hasan
- Beatrice Nathania
- Callista Giana Then
- Davina Oktaviani
- Muhammad Shafi Dhihar Athaya

In [ ]:
import os
import random
import shutil
from ultralytics import YOLO
from collections import Counter

Berdasarkan data-data yang ada, kita perlu melakukan remapping agar sesuai dengan format sebagai berikut:
0 = cigarette
1 = person
2 = smoke
3 = vape.
Maka dari itu, data 4, 5, 6, dan 7 perlu dilakukan remap karena indeks keempat dataset tersebut berbeda.

## Data 4

In [ ]:
data4_label = "data4/train/labels"

for file in os.listdir(data4_label):
    path = os.path.join(data4_label, file)

    with open(path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split()
        class_id = int(parts[0])

        if class_id == 0:
            parts[0] = "0"
        elif class_id == 1:
            parts[0] = "3"

        new_lines.append(" ".join(parts))

    with open(path, "w") as f:
        f.write("\n".join(new_lines))

## Data 5

In [ ]:
data5_label = "data5/train/labels"

for file in os.listdir(data5_label):
    path = os.path.join(data5_label, file)

    with open(path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split()
        class_id = int(parts[0])

        if class_id == 0:
            parts[0] = "2"  #smoke
        elif class_id == 1:
            parts[0] = "3"  #vape
        elif class_id == 2:
            parts[0] = "1"  #person

        new_lines.append(" ".join(parts))

    with open(path, "w") as f:
        f.write("\n".join(new_lines))

## Data 6

In [ ]:
data6_label = "data6/train/labels"

for file in os.listdir(data6_label):
    path = os.path.join(data6_label, file)

    with open(path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split()
        class_id = int(parts[0])

        if class_id == 0:
            parts[0] = "2"  #smoke
        elif class_id == 1:
            parts[0] = "3"  #vape
        elif class_id == 2:
            parts[0] = "1"  #person

        new_lines.append(" ".join(parts))

    with open(path, "w") as f:
        f.write("\n".join(new_lines))

## Data 7

In [ ]:
data7_label = "data7/train/labels"

for file in os.listdir(data7_label):
    path = os.path.join(data7_label, file)

    with open(path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split()
        class_id = int(parts[0])

        if class_id == 0:
            parts[0] = "0"
        elif class_id == 1:
            parts[0] = "3"

        new_lines.append(" ".join(parts))

    with open(path, "w") as f:
        f.write("\n".join(new_lines))

# Splitting Dataset (Data3, Data4, Data5, Data 7)

Dataset data3, data4, data5, dan data7 hanya berisi file training. Split menjadi train, validation, dan test dengan proposi 70:15:15

In [28]:
def split_dataset(src_dir, dest_dir, train_ratio=0.7, val_ratio=0.15):
    img_dir = os.path.join(src_dir, "train/images")
    label_dir = os.path.join(src_dir, "train/labels")

    images = os.listdir(img_dir)
    random.shuffle(images)

    total = len(images)
    train_end = int(total * train_ratio)
    val_end = int(total * (train_ratio + val_ratio))

    train_imgs = images[:train_end]
    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]

    for split in ["train", "valid", "test"]:
        for sub in ["images", "labels"]:
            os.makedirs(os.path.join(dest_dir, split, sub), exist_ok=True)

    def copy_files(img_list, split):
        for img in img_list:
            shutil.copy(
                os.path.join(img_dir, img),
                os.path.join(dest_dir, split, "images", img)
            )

            label = os.path.splitext(img)[0] + ".txt"

            shutil.copy(
                os.path.join(label_dir, label),
                os.path.join(dest_dir, split, "labels", label)
            )

    copy_files(train_imgs, "train")
    copy_files(val_imgs, "valid")
    copy_files(test_imgs, "test")

    print(f"{src_dir} DONE:")
    print(f"Train: {len(train_imgs)}")
    print(f"Valid: {len(val_imgs)}")
    print(f"Test : {len(test_imgs)}")

In [29]:
split_dataset("data4", "data4_split")
split_dataset("data5", "data5_split")
split_dataset("data7", "data7_split")
split_dataset("data3", "data3_split")

data4 DONE:
Train: 1153
Valid: 247
Test : 248
data5 DONE:
Train: 44
Valid: 9
Test : 10
data7 DONE:
Train: 70
Valid: 15
Test : 15
data3 DONE:
Train: 330
Valid: 71
Test : 71


# Merge Dataset

In [30]:
def merge_datasets(src_dirs, dest_dir):
    splits = ['test', "train", "valid"]

    for split in splits:
        for sub in ["images", "labels"]:
            os.makedirs(os.path.join(dest_dir, split, sub), exist_ok=True)

    for src in src_dirs:
        for split in splits:
            for sub in ["images", "labels"]:
                src_path = os.path.join(src, split, sub)
                dest_path = os.path.join(dest_dir, split, sub)

                if not os.path.exists(src_path):
                    continue

                for file in os.listdir(src_path):
                    src_file = os.path.join(src_path, file)
                    new_name = src.split("/")[-1] + "_" + file
                    dest_file = os.path.join(dest_path, new_name)
                    shutil.copy(src_file, dest_file)

In [31]:
merge_datasets(
    [
        "data1",
        "data2",
        "data3_split",
        "data4_split",
        "data5_split",
        "data6",
        "data7_split"
    ],
    "dataset_final"
)

In [32]:
label_path = "dataset_final/train/labels"
counter = Counter()

for file in os.listdir(label_path):
    with open(os.path.join(label_path, file)) as f:
        for line in f:
            class_id = int(line.split()[0])
            counter[class_id] += 1

print(counter)

Counter({0: 4971, 1: 4895, 3: 2114, 2: 857})


In [33]:
label_path = "dataset_final/test/labels"
counter = Counter()

for file in os.listdir(label_path):
    with open(os.path.join(label_path, file)) as f:
        for line in f:
            class_id = int(line.split()[0])
            counter[class_id] += 1

print(counter)

Counter({0: 956, 1: 915, 3: 470, 2: 166})


In [34]:
label_path = "dataset_final/valid/labels"
counter = Counter()

for file in os.listdir(label_path):
    with open(os.path.join(label_path, file)) as f:
        for line in f:
            class_id = int(line.split()[0])
            counter[class_id] += 1

print(counter)

Counter({1: 1055, 0: 1053, 3: 463, 2: 199})


Total data:
6.980 data rokok, 6.865 data person, 3.047 data vape, dan 1.222 data smoke.